In [1]:
from spark_utils import get_spark

spark = get_spark("SparkSQLQueries")
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("data/silver/dados_limpos.csv")
)
df.createOrReplaceTempView("projeto_final")
print(f"View 'projeto_final' registrada com {df.count()} linhas.")

resultado = spark.sql("""
SELECT COUNT(*) as total_registros
FROM projeto_final
""")
resultado.show()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/11/10 20:29:43 WARN Utils: Your hostname, Davis-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.200.60.101 instead (on interface en0)
25/11/10 20:29:43 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/10 20:29:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/11/10 20:29:44 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/11/10 20:29:44 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
25/11/10 20:29:44 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
25/11/10 20:29:44 WA

View 'projeto_final' registrada com 10476 linhas.
+---------------+
|total_registros|
+---------------+
|          10476|
+---------------+



In [2]:
query_top_rendas = spark.sql("""
SELECT CODIGO_CLIENTE, RENDA_TOTAL
FROM projeto_final
ORDER BY RENDA_TOTAL DESC
LIMIT 10
""")
query_top_rendas.show()


+--------------+-----------+
|CODIGO_CLIENTE|RENDA_TOTAL|
+--------------+-----------+
|           144|    24400.0|
|          3037|    24400.0|
|          5580|    24400.0|
|          7344|    24400.0|
|          8316|    24400.0|
|          3684|    24400.0|
|          1291|    24400.0|
|          2024|    24400.0|
|          6854|    24400.0|
|          6877|    24400.0|
+--------------+-----------+



In [3]:
score_faixas = spark.sql("""
SELECT
    CASE
        WHEN SCORE < 25 THEN 'Baixo (0-24)'
        WHEN SCORE BETWEEN 25 AND 49 THEN 'Regular (25-49)'
        WHEN SCORE BETWEEN 50 AND 74 THEN 'Bom (50-74)'
        ELSE 'Excelente (75-100)'
    END AS faixa_score,
    COUNT(*) AS total_clientes
FROM projeto_final
GROUP BY faixa_score
ORDER BY total_clientes DESC
""")
score_faixas.show()


+------------------+--------------+
|       faixa_score|total_clientes|
+------------------+--------------+
|   Regular (25-49)|          2988|
|Excelente (75-100)|          2610|
|       Bom (50-74)|          2448|
|      Baixo (0-24)|          2430|
+------------------+--------------+



In [4]:
carros_renda = spark.sql("""
SELECT
    QT_CARROS,
    AVG(RENDA_TOTAL) AS renda_media,
    AVG(ULTIMO_SALARIO) AS salario_medio,
    AVG(SCORE) AS score_medio
FROM projeto_final
GROUP BY QT_CARROS
ORDER BY QT_CARROS
""")
carros_renda.show()


+---------+------------------+------------------+-----------------+
|QT_CARROS|       renda_media|     salario_medio|      score_medio|
+---------+------------------+------------------+-----------------+
|        0| 6467.278043593833| 5535.220627325891|47.36842105263158|
|        1|14050.746268656716|13740.298507462687| 53.6318407960199|
|        2| 5930.813953488372| 5256.395348837209|51.47093023255814|
+---------+------------------+------------------+-----------------+



In [5]:
trabalho_renda = spark.sql("""
SELECT
    TRABALHANDO_ATUALMENTE,
    COUNT(*) AS total_clientes,
    AVG(RENDA_TOTAL) AS renda_media,
    AVG(ULTIMO_SALARIO) AS salario_medio,
    AVG(SCORE) AS score_medio
FROM projeto_final
GROUP BY TRABALHANDO_ATUALMENTE
""")
trabalho_renda.show()


+----------------------+--------------+-----------------+-----------------+------------------+
|TRABALHANDO_ATUALMENTE|total_clientes|      renda_media|    salario_medio|       score_medio|
+----------------------+--------------+-----------------+-----------------+------------------+
|                  true|          6336|8797.616792929293|8001.025883838384| 45.05681818181818|
|                 false|          4140| 9126.95652173913|8723.478260869566|59.447826086956525|
+----------------------+--------------+-----------------+-----------------+------------------+

